In [ ]:
import os
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import cv2
import pydicom
import torch

print("Everything imported successfully!")
print("GPU available:", torch.cuda.is_available())

if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))

print("\nFiles available in /kaggle/input:")
for item in os.listdir("/kaggle/input"):
    print(item)

In [ ]:
DATA_DIR = "/kaggle/input/competitions/rsna-pneumonia-detection-challenge"

print("Dataset contents:\n")

for item in os.listdir(DATA_DIR):
    print(item)

In [ ]:
import os

print("Folders available in /kaggle/input:")
print(os.listdir("/kaggle/input"))
import os

print(os.listdir("/kaggle/input/competitions"))

In [ ]:
DATA_DIR = "/kaggle/input/competitions/rsna-pneumonia-detection-challenge"

import os

print("Dataset contents:")
for item in os.listdir(DATA_DIR):
    print(item)

In [ ]:
import pandas as pd

labels = pd.read_csv(
    DATA_DIR + "/stage_2_train_labels.csv"
)

class_info = pd.read_csv(
    DATA_DIR + "/stage_2_detailed_class_info.csv"
)

print("Labels shape:", labels.shape)
print("Class info shape:", class_info.shape)

print("\nClass distribution:")
print(class_info["class"].value_counts())

In [ ]:
import os
import pydicom
import matplotlib.pyplot as plt

IMAGE_DIR = DATA_DIR + "/stage_2_train_images"

# Pick the first patient ID from the class information
patient_id = class_info.iloc[0]["patientId"]

# DICOM filename
dicom_path = os.path.join(IMAGE_DIR, patient_id + ".dcm")

print("Patient ID:", patient_id)
print("File exists:", os.path.exists(dicom_path))
print("Path:", dicom_path)

# Read DICOM
dcm = pydicom.dcmread(dicom_path)

# Get image
image = dcm.pixel_array

print("Image shape:", image.shape)

# Display
plt.figure(figsize=(8, 8))
plt.imshow(image, cmap="gray")
plt.axis("off")
plt.title("RSNA Chest X-ray")
plt.show()

In [ ]:
# ============================================
# CREATE OUR BINARY DATASET
# ============================================

# Keep only the two classes we want
binary_df = class_info[
    class_info["class"].isin(["Lung Opacity", "Normal"])
].copy()

# Create binary labels
# Lung Opacity = 1
# Normal = 0
binary_df["label"] = (
    binary_df["class"] == "Lung Opacity"
).astype(int)

# Show the result
print("Total images:", len(binary_df))

print("\nClass distribution:")
print(binary_df["class"].value_counts())

print("\nLabel distribution:")
print(binary_df["label"].value_counts())

display(binary_df.head())

In [ ]:
# Check whether the corresponding DICOM files exist

IMAGE_DIR = DATA_DIR + "/stage_2_train_images"

binary_df["dicom_path"] = binary_df["patientId"].apply(
    lambda x: os.path.join(IMAGE_DIR, x + ".dcm")
)

binary_df["file_exists"] = binary_df["dicom_path"].apply(
    os.path.exists
)

print("Files found:", binary_df["file_exists"].sum())
print("Files missing:", (~binary_df["file_exists"]).sum())

In [ ]:
# ============================================
# DISPLAY A PNEUMONIA IMAGE WITH GROUND TRUTH
# ============================================

import matplotlib.patches as patches

# Pick a pneumonia example that has a bounding box
pneumonia_id = labels[labels["Target"] == 1].iloc[0]["patientId"]

# Get its DICOM
dicom_path = os.path.join(
    IMAGE_DIR,
    pneumonia_id + ".dcm"
)

dcm = pydicom.dcmread(dicom_path)
image = dcm.pixel_array

# Get all bounding boxes for this patient
boxes = labels[
    (labels["patientId"] == pneumonia_id) &
    (labels["Target"] == 1)
]

# Display
fig, ax = plt.subplots(figsize=(8, 8))

ax.imshow(image, cmap="gray")

for _, row in boxes.iterrows():
    rect = patches.Rectangle(
        (row["x"], row["y"]),
        row["width"],
        row["height"],
        linewidth=2,
        edgecolor="red",
        facecolor="none"
    )
    ax.add_patch(rect)

ax.set_title("RSNA Pneumonia Image + Ground Truth")
ax.axis("off")

plt.show()

print("Patient ID:", pneumonia_id)
print("Number of bounding boxes:", len(boxes))

In [ ]:
# ============================================
# CREATE A BALANCED 6000-IMAGE DATASET
# ============================================

from sklearn.model_selection import train_test_split

# 3000 samples from each class
normal_df = binary_df[binary_df["label"] == 0].sample(
    n=3000,
    random_state=42
)

pneumonia_df = binary_df[binary_df["label"] == 1].sample(
    n=3000,
    random_state=42
)

# Combine them
project_df = pd.concat(
    [normal_df, pneumonia_df]
).sample(
    frac=1,
    random_state=42
).reset_index(drop=True)

print("Total project images:", len(project_df))
print("\nClass distribution:")
print(project_df["label"].value_counts())

In [ ]:
# ============================================
# TRAIN / VALIDATION / TEST SPLIT
# ============================================

train_df, temp_df = train_test_split(
    project_df,
    test_size=0.30,
    stratify=project_df["label"],
    random_state=42
)

val_df, test_df = train_test_split(
    temp_df,
    test_size=0.50,
    stratify=temp_df["label"],
    random_state=42
)

print("Training images:", len(train_df))
print("Validation images:", len(val_df))
print("Test images:", len(test_df))

print("\nTraining distribution:")
print(train_df["label"].value_counts())

print("\nValidation distribution:")
print(val_df["label"].value_counts())

print("\nTest distribution:")
print(test_df["label"].value_counts())

In [ ]:
# ============================================
# REMOVE DUPLICATE PATIENT IDs
# Keep ONE row per X-ray
# ============================================

binary_unique_df = binary_df.drop_duplicates(
    subset="patientId"
).copy()

print("Rows before removing duplicates:", len(binary_df))
print("Unique X-ray images:", len(binary_unique_df))

print("\nClass distribution:")
print(binary_unique_df["class"].value_counts())

print("\nLabel distribution:")
print(binary_unique_df["label"].value_counts())
print(
    "Duplicate patient IDs:",
    binary_unique_df["patientId"].duplicated().sum()
)

In [ ]:
binary_unique_df = binary_df.drop_duplicates(
    subset="patientId"
).copy()

print("Rows before removing duplicates:", len(binary_df))
print("Unique X-ray images:", len(binary_unique_df))

print("\nClass distribution:")
print(binary_unique_df["class"].value_counts())

print("\nLabel distribution:")
print(binary_unique_df["label"].value_counts())

In [ ]:
print(
    "Duplicate patient IDs:",
    binary_unique_df["patientId"].duplicated().sum()
)

In [ ]:
# ============================================
# CREATE BALANCED DATASET
# ============================================

normal_df = binary_unique_df[
    binary_unique_df["label"] == 0
].sample(n=3000, random_state=42)

opacity_df = binary_unique_df[
    binary_unique_df["label"] == 1
].sample(n=3000, random_state=42)

project_df = pd.concat(
    [normal_df, opacity_df]
).sample(
    frac=1,
    random_state=42
).reset_index(drop=True)

print("Total images:", len(project_df))
print("\nClass distribution:")
print(project_df["label"].value_counts())

In [ ]:
# ============================================
# TRAIN / VALIDATION / TEST SPLIT
# ============================================

from sklearn.model_selection import train_test_split

train_df, temp_df = train_test_split(
    project_df,
    test_size=0.30,
    stratify=project_df["label"],
    random_state=42
)

val_df, test_df = train_test_split(
    temp_df,
    test_size=0.50,
    stratify=temp_df["label"],
    random_state=42
)

print("Training:", len(train_df))
print("Validation:", len(val_df))
print("Testing:", len(test_df))

print("\nTraining:")
print(train_df["label"].value_counts())

print("\nValidation:")
print(val_df["label"].value_counts())

print("\nTesting:")
print(test_df["label"].value_counts())

In [ ]:
# ============================================
# PYTORCH DATASET
# ============================================

import torch
from torch.utils.data import Dataset, DataLoader
from torchvision import transforms
from PIL import Image

class RSNADataset(Dataset):

    def __init__(self, dataframe, image_dir, transform=None):
        self.dataframe = dataframe.reset_index(drop=True)
        self.image_dir = image_dir
        self.transform = transform

    def __len__(self):
        return len(self.dataframe)

    def __getitem__(self, index):

        row = self.dataframe.iloc[index]

        patient_id = row["patientId"]
        label = int(row["label"])

        # DICOM path
        path = os.path.join(
            self.image_dir,
            patient_id + ".dcm"
        )

        # Read DICOM
        dcm = pydicom.dcmread(path)

        # Convert to pixel array
        image = dcm.pixel_array.astype(np.float32)

        # Normalize image to 0-255
        image = image - image.min()

        if image.max() > 0:
            image = image / image.max()

        image = (image * 255).astype(np.uint8)

        # Convert grayscale → RGB
        image = Image.fromarray(image).convert("RGB")

        # Apply transforms
        if self.transform:
            image = self.transform(image)

        return image, torch.tensor(label, dtype=torch.long)

In [ ]:
# ============================================
# IMAGE TRANSFORMS
# ============================================

train_transform = transforms.Compose([
    transforms.Resize((224, 224)),
    
    transforms.RandomHorizontalFlip(p=0.5),
    transforms.RandomRotation(5),
    
    transforms.ToTensor(),
    
    transforms.Normalize(
        mean=[0.485, 0.456, 0.406],
        std=[0.229, 0.224, 0.225]
    )
])

test_transform = transforms.Compose([
    transforms.Resize((224, 224)),
    
    transforms.ToTensor(),
    
    transforms.Normalize(
        mean=[0.485, 0.456, 0.406],
        std=[0.229, 0.224, 0.225]
    )
])

In [ ]:
# ============================================
# CREATE DATASETS
# ============================================

train_dataset = RSNADataset(
    train_df,
    IMAGE_DIR,
    train_transform
)

val_dataset = RSNADataset(
    val_df,
    IMAGE_DIR,
    test_transform
)

test_dataset = RSNADataset(
    test_df,
    IMAGE_DIR,
    test_transform
)

# DataLoaders

train_loader = DataLoader(
    train_dataset,
    batch_size=32,
    shuffle=True,
    num_workers=2,
    pin_memory=True
)

val_loader = DataLoader(
    val_dataset,
    batch_size=32,
    shuffle=False,
    num_workers=2,
    pin_memory=True
)

test_loader = DataLoader(
    test_dataset,
    batch_size=32,
    shuffle=False,
    num_workers=2,
    pin_memory=True
)

print("Datasets created!")
print("Training batches:", len(train_loader))
print("Validation batches:", len(val_loader))
print("Testing batches:", len(test_loader))

In [ ]:
# ============================================
# TEST DATA LOADING
# ============================================

images, labels_batch = next(iter(train_loader))

print("Image batch shape:", images.shape)
print("Label batch shape:", labels_batch.shape)
print("Labels:", labels_batch[:10])

In [ ]:
# ============================================
# DENSENET121
# ============================================

import torch.nn as nn
from torchvision.models import densenet121, DenseNet121_Weights

device = torch.device(
    "cuda" if torch.cuda.is_available() else "cpu"
)

print("Using:", device)

# Load pretrained DenseNet121
weights = DenseNet121_Weights.DEFAULT

model = densenet121(weights=weights)

# Replace final classifier
num_features = model.classifier.in_features

model.classifier = nn.Linear(
    num_features,
    2
)

model = model.to(device)

print(model.classifier)

In [ ]:
# ============================================
# LOSS + OPTIMIZER
# ============================================

import torch.optim as optim

criterion = nn.CrossEntropyLoss()

optimizer = optim.Adam(
    model.parameters(),
    lr=0.0001
)

print("Loss function:", criterion)
print("Optimizer:", optimizer)

In [ ]:
# ============================================
# SMALLER DATASET FOR FAST TRAINING
# ============================================

normal_df = binary_unique_df[
    binary_unique_df["label"] == 0
].sample(n=1000, random_state=42)

opacity_df = binary_unique_df[
    binary_unique_df["label"] == 1
].sample(n=1000, random_state=42)

project_df = pd.concat(
    [normal_df, opacity_df]
).sample(
    frac=1,
    random_state=42
).reset_index(drop=True)

print("Total images:", len(project_df))
print(project_df["label"].value_counts())

In [ ]:
from sklearn.model_selection import train_test_split

train_df, temp_df = train_test_split(
    project_df,
    test_size=0.30,
    stratify=project_df["label"],
    random_state=42
)

val_df, test_df = train_test_split(
    temp_df,
    test_size=0.50,
    stratify=temp_df["label"],
    random_state=42
)

print("Training:", len(train_df))
print("Validation:", len(val_df))
print("Testing:", len(test_df))

In [ ]:
train_dataset = RSNADataset(
    train_df,
    IMAGE_DIR,
    train_transform
)

val_dataset = RSNADataset(
    val_df,
    IMAGE_DIR,
    test_transform
)

test_dataset = RSNADataset(
    test_df,
    IMAGE_DIR,
    test_transform
)

train_loader = DataLoader(
    train_dataset,
    batch_size=32,
    shuffle=True,
    num_workers=4,
    pin_memory=True
)

val_loader = DataLoader(
    val_dataset,
    batch_size=32,
    shuffle=False,
    num_workers=4,
    pin_memory=True
)

test_loader = DataLoader(
    test_dataset,
    batch_size=32,
    shuffle=False,
    num_workers=4,
    pin_memory=True
)

print("Train batches:", len(train_loader))
print("Validation batches:", len(val_loader))
print("Test batches:", len(test_loader))

In [ ]:
# ============================================
# FREEZE DENSENET FEATURES
# ============================================

for param in model.features.parameters():
    param.requires_grad = False

# Only the final classifier will train
for param in model.classifier.parameters():
    param.requires_grad = True

print("DenseNet features frozen.")
print("Classifier will be trained.")

In [ ]:
optimizer = optim.Adam(
    model.classifier.parameters(),
    lr=0.001
)

criterion = nn.CrossEntropyLoss()

print("Ready for fast training!")

In [ ]:
num_epochs = 2

for epoch in range(num_epochs):

    model.train()

    running_loss = 0.0
    correct = 0
    total = 0

    for images, labels_batch in train_loader:

        images = images.to(device, non_blocking=True)
        labels_batch = labels_batch.to(device, non_blocking=True)

        optimizer.zero_grad()

        outputs = model(images)

        loss = criterion(outputs, labels_batch)

        loss.backward()

        optimizer.step()

        running_loss += loss.item()

        _, predicted = torch.max(outputs, 1)

        total += labels_batch.size(0)
        correct += (predicted == labels_batch).sum().item()

    train_accuracy = 100 * correct / total

    # Validation
    model.eval()

    val_correct = 0
    val_total = 0

    with torch.no_grad():

        for images, labels_batch in val_loader:

            images = images.to(device, non_blocking=True)
            labels_batch = labels_batch.to(device, non_blocking=True)

            outputs = model(images)

            _, predicted = torch.max(outputs, 1)

            val_total += labels_batch.size(0)
            val_correct += (
                predicted == labels_batch
            ).sum().item()

    val_accuracy = 100 * val_correct / val_total

    print(
        f"Epoch {epoch+1}/{num_epochs} | "
        f"Loss: {running_loss/len(train_loader):.4f} | "
        f"Train Accuracy: {train_accuracy:.2f}% | "
        f"Validation Accuracy: {val_accuracy:.2f}%"
    )

In [ ]:
from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    roc_auc_score,
    confusion_matrix,
    classification_report
)

model.eval()

all_labels = []
all_predictions = []
all_probabilities = []

with torch.no_grad():

    for images, labels_batch in test_loader:

        images = images.to(device)
        labels_batch = labels_batch.to(device)

        outputs = model(images)

        probabilities = torch.softmax(outputs, dim=1)
        predictions = torch.argmax(outputs, dim=1)

        all_labels.extend(labels_batch.cpu().numpy())
        all_predictions.extend(predictions.cpu().numpy())
        all_probabilities.extend(
            probabilities[:, 1].cpu().numpy()
        )

# Metrics
accuracy = accuracy_score(
    all_labels,
    all_predictions
)

precision = precision_score(
    all_labels,
    all_predictions
)

recall = recall_score(
    all_labels,
    all_predictions
)

f1 = f1_score(
    all_labels,
    all_predictions
)

auc = roc_auc_score(
    all_labels,
    all_probabilities
)

print("========== TEST RESULTS ==========")
print(f"Accuracy : {accuracy:.4f}")
print(f"Precision: {precision:.4f}")
print(f"Recall   : {recall:.4f}")
print(f"F1 Score : {f1:.4f}")
print(f"ROC-AUC  : {auc:.4f}")

print("\nClassification Report:")
print(
    classification_report(
        all_labels,
        all_predictions,
        target_names=["Normal", "Lung Opacity"]
    )
)

In [ ]:
import seaborn as sns

cm = confusion_matrix(
    all_labels,
    all_predictions
)

plt.figure(figsize=(6, 5))

sns.heatmap(
    cm,
    annot=True,
    fmt="d",
    cmap="Blues",
    xticklabels=["Normal", "Lung Opacity"],
    yticklabels=["Normal", "Lung Opacity"]
)

plt.xlabel("Predicted")
plt.ylabel("Actual")
plt.title("Confusion Matrix")

plt.show()

In [ ]:
# ============================================
# GRAD-CAM
# ============================================

class GradCAM:

    def __init__(self, model, target_layer):

        self.model = model
        self.target_layer = target_layer

        self.gradients = None
        self.activations = None

        # Forward hook
        target_layer.register_forward_hook(
            self.save_activation
        )

        # Backward hook
        target_layer.register_full_backward_hook(
            self.save_gradient
        )

    def save_activation(
        self,
        module,
        input,
        output
    ):
        self.activations = output

    def save_gradient(
        self,
        module,
        grad_input,
        grad_output
    ):
        self.gradients = grad_output[0]

    def generate(
        self,
        input_tensor,
        class_index
    ):

        self.model.zero_grad()

        output = self.model(input_tensor)

        score = output[:, class_index]

        score.backward()

        # Gradients
        gradients = self.gradients

        # Feature maps
        activations = self.activations

        # Global average pooling of gradients
        weights = gradients.mean(
            dim=(2, 3),
            keepdim=True
        )

        # Weighted feature maps
        cam = (
            weights * activations
        ).sum(dim=1)

        # ReLU
        cam = torch.relu(cam)

        # Normalize
        cam = cam - cam.min()

        if cam.max() > 0:
            cam = cam / cam.max()

        return cam.detach().cpu().numpy()[0]

In [ ]:
target_layer = model.features.denseblock4

gradcam = GradCAM(
    model,
    target_layer
)

print("Grad-CAM initialized!")

In [ ]:
# ============================================
# SELECT TEST IMAGE
# ============================================

row = test_df[
    test_df["label"] == 1
].iloc[0]

patient_id = row["patientId"]
true_label = row["label"]

dicom_path = os.path.join(
    IMAGE_DIR,
    patient_id + ".dcm"
)

dcm = pydicom.dcmread(dicom_path)

original_image = dcm.pixel_array.astype(
    np.float32
)

# Normalize for display
display_image = original_image.copy()

display_image -= display_image.min()

if display_image.max() > 0:
    display_image /= display_image.max()

print("Patient ID:", patient_id)
print("True label:", true_label)
print("Image shape:", original_image.shape)

In [ ]:
# Convert to 8-bit
image_uint8 = (
    display_image * 255
).astype(np.uint8)

# Grayscale → RGB
pil_image = Image.fromarray(
    image_uint8
).convert("RGB")

# Apply test transform
input_tensor = test_transform(
    pil_image
).unsqueeze(0).to(device)

print("Model input shape:", input_tensor.shape)

In [ ]:
# ============================================
# PREDICTION + GRAD-CAM
# ============================================

model.eval()

# Enable gradients
input_tensor.requires_grad_()

output = model(input_tensor)

probabilities = torch.softmax(
    output,
    dim=1
)

prediction = torch.argmax(
    probabilities,
    dim=1
).item()

confidence = probabilities[
    0, prediction
].item()

print(
    "Prediction:",
    "Lung Opacity" if prediction == 1 else "Normal"
)

print(
    "Confidence:",
    f"{confidence*100:.2f}%"
)

# Generate CAM
cam = gradcam.generate(
    input_tensor,
    prediction
)

print("CAM shape:", cam.shape)

In [ ]:
# ============================================
# DISPLAY GRAD-CAM
# ============================================

import cv2

# Resize CAM to original image size
cam_resized = cv2.resize(
    cam,
    (
        original_image.shape[1],
        original_image.shape[0]
    )
)

plt.figure(figsize=(12, 5))

# Original
plt.subplot(1, 2, 1)

plt.imshow(
    display_image,
    cmap="gray"
)

plt.title(
    "Original X-ray"
)

plt.axis("off")

# Heatmap
plt.subplot(1, 2, 2)

plt.imshow(
    display_image,
    cmap="gray"
)

plt.imshow(
    cam_resized,
    cmap="jet",
    alpha=0.45
)

plt.title(
    "Grad-CAM Heatmap"
)

plt.axis("off")

plt.show()

In [ ]:
# ============================================
# GET RSNA GROUND-TRUTH BOXES
# ============================================

ground_truth_boxes = labels[
    (labels["patientId"] == patient_id) &
    (labels["Target"] == 1)
]

print("Patient ID:", patient_id)
print("Number of ground-truth boxes:", len(ground_truth_boxes))

display(
    ground_truth_boxes[
        ["x", "y", "width", "height", "Target"]
    ]
)

In [ ]:
# ============================================
# CREATE PREDICTED BOUNDING BOX FROM GRAD-CAM
# ============================================

# Threshold Grad-CAM
threshold = 0.5

binary_cam = (
    cam_resized >= threshold
).astype(np.uint8) * 255

# Find contours
contours, _ = cv2.findContours(
    binary_cam,
    cv2.RETR_EXTERNAL,
    cv2.CHAIN_APPROX_SIMPLE
)

# Find the largest activated region
if len(contours) > 0:

    largest_contour = max(
        contours,
        key=cv2.contourArea
    )

    predicted_box = cv2.boundingRect(
        largest_contour
    )

else:

    predicted_box = (
        0,
        0,
        0,
        0
    )

print(
    "Predicted Grad-CAM box:",
    predicted_box
)

In [ ]:
# ============================================
# IoU FUNCTION
# ============================================

def calculate_iou(box1, box2):

    x1, y1, w1, h1 = box1
    x2, y2, w2, h2 = box2

    # Coordinates of intersection
    left = max(x1, x2)
    top = max(y1, y2)

    right = min(
        x1 + w1,
        x2 + w2
    )

    bottom = min(
        y1 + h1,
        y2 + h2
    )

    # Intersection area
    if right <= left or bottom <= top:
        intersection = 0
    else:
        intersection = (
            (right - left) *
            (bottom - top)
        )

    # Individual areas
    area1 = w1 * h1
    area2 = w2 * h2

    # Union
    union = area1 + area2 - intersection

    if union == 0:
        return 0

    return intersection / union

In [ ]:
# ============================================
# CALCULATE BEST IoU
# ============================================

iou_scores = []

for _, row in ground_truth_boxes.iterrows():

    gt_box = (
        int(row["x"]),
        int(row["y"]),
        int(row["width"]),
        int(row["height"])
    )

    iou = calculate_iou(
        predicted_box,
        gt_box
    )

    iou_scores.append(iou)

if len(iou_scores) > 0:

    best_iou = max(iou_scores)

else:

    best_iou = 0

print("IoU scores:", iou_scores)
print(f"Best IoU: {best_iou:.4f}")

In [ ]:
# ============================================
# FINAL LOCALIZATION VISUALIZATION
# ============================================

fig, ax = plt.subplots(figsize=(8, 8))

ax.imshow(
    display_image,
    cmap="gray"
)

# Ground truth boxes - RED
for _, row in ground_truth_boxes.iterrows():

    gt_rect = patches.Rectangle(
        (row["x"], row["y"]),
        row["width"],
        row["height"],
        linewidth=3,
        edgecolor="red",
        facecolor="none",
        label="Ground Truth"
    )

    ax.add_patch(gt_rect)

# Predicted box - GREEN
px, py, pw, ph = predicted_box

pred_rect = patches.Rectangle(
    (px, py),
    pw,
    ph,
    linewidth=3,
    edgecolor="lime",
    facecolor="none",
    label="Grad-CAM Prediction"
)

ax.add_patch(pred_rect)

ax.set_title(
    f"Pneumonia Localization\nIoU = {best_iou:.3f}"
)

ax.legend()
ax.axis("off")

plt.show()

In [ ]:
print("IoU scores:", iou_scores)
print(f"Best IoU: {best_iou:.4f}")

In [ ]:
# ============================================
# SAVE TRAINED MODEL
# ============================================

torch.save(
    model.state_dict(),
    "/kaggle/working/densenet121_pneumonia.pth"
)

print("Model saved successfully!")

In [ ]:
# ============================================
# SAVE FINAL LOCALIZATION FIGURE
# ============================================

fig, ax = plt.subplots(figsize=(8, 8))

ax.imshow(
    display_image,
    cmap="gray"
)

# Ground truth - RED
for _, row in ground_truth_boxes.iterrows():

    gt_rect = patches.Rectangle(
        (row["x"], row["y"]),
        row["width"],
        row["height"],
        linewidth=3,
        edgecolor="red",
        facecolor="none",
        label="Ground Truth"
    )

    ax.add_patch(gt_rect)

# Grad-CAM prediction - GREEN
px, py, pw, ph = predicted_box

pred_rect = patches.Rectangle(
    (px, py),
    pw,
    ph,
    linewidth=3,
    edgecolor="lime",
    facecolor="none",
    label="Grad-CAM Prediction"
)

ax.add_patch(pred_rect)

ax.set_title(
    f"Pneumonia Localization\nIoU = {best_iou:.3f}"
)

ax.legend()
ax.axis("off")

plt.tight_layout()

plt.savefig(
    "/kaggle/working/final_localization.png",
    dpi=300,
    bbox_inches="tight"
)

plt.show()

print("Figure saved!")

In [ ]:
plt.figure(figsize=(8, 8))

plt.imshow(
    display_image,
    cmap="gray"
)

plt.imshow(
    cam_resized,
    cmap="jet",
    alpha=0.45
)

plt.title(
    f"Grad-CAM Heatmap - Confidence {confidence*100:.2f}%"
)

plt.axis("off")

plt.tight_layout()

plt.savefig(
    "/kaggle/working/gradcam_heatmap.png",
    dpi=300,
    bbox_inches="tight"
)

plt.show()

print("Grad-CAM figure saved!")

In [ ]:
# ============================================
# EVALUATE GRAD-CAM LOCALIZATION ON 20 IMAGES
# ============================================

import random

pneumonia_test = test_df[
    test_df["label"] == 1
].copy()

# Select 20 pneumonia cases
sample_rows = pneumonia_test.sample(
    n=20,
    random_state=42
)

all_ious = []

for idx, (_, row) in enumerate(sample_rows.iterrows()):

    patient_id_temp = row["patientId"]

    # DICOM
    dicom_path_temp = os.path.join(
        IMAGE_DIR,
        patient_id_temp + ".dcm"
    )

    dcm_temp = pydicom.dcmread(dicom_path_temp)

    image_temp = dcm_temp.pixel_array.astype(
        np.float32
    )

    # Normalize
    image_temp = image_temp - image_temp.min()

    if image_temp.max() > 0:
        image_temp = image_temp / image_temp.max()

    image_uint8_temp = (
        image_temp * 255
    ).astype(np.uint8)

    pil_temp = Image.fromarray(
        image_uint8_temp
    ).convert("RGB")

    # Model input
    input_temp = test_transform(
        pil_temp
    ).unsqueeze(0).to(device)

    # Grad-CAM
    model.zero_grad()

    output_temp = model(input_temp)

    prediction_temp = torch.argmax(
        output_temp,
        dim=1
    ).item()

    # Only localize if predicted as opacity
    if prediction_temp != 1:
        continue

    cam_temp = gradcam.generate(
        input_temp,
        prediction_temp
    )

    cam_temp_resized = cv2.resize(
        cam_temp,
        (
            image_temp.shape[1],
            image_temp.shape[0]
        )
    )

    # Threshold
    binary_temp = (
        cam_temp_resized >= 0.5
    ).astype(np.uint8) * 255

    contours_temp, _ = cv2.findContours(
        binary_temp,
        cv2.RETR_EXTERNAL,
        cv2.CHAIN_APPROX_SIMPLE
    )

    if len(contours_temp) == 0:
        continue

    largest_temp = max(
        contours_temp,
        key=cv2.contourArea
    )

    predicted_box_temp = cv2.boundingRect(
        largest_temp
    )

    # Ground truth
    gt_temp = labels[
        (labels["patientId"] == patient_id_temp) &
        (labels["Target"] == 1)
    ]

    patient_ious = []

    for _, gt_row in gt_temp.iterrows():

        gt_box_temp = (
            int(gt_row["x"]),
            int(gt_row["y"]),
            int(gt_row["width"]),
            int(gt_row["height"])
        )

        iou_temp = calculate_iou(
            predicted_box_temp,
            gt_box_temp
        )

        patient_ious.append(iou_temp)

    if patient_ious:
        all_ious.append(max(patient_ious))

print("Images successfully localized:", len(all_ious))

if all_ious:
    print(
        "Mean IoU:",
        round(np.mean(all_ious), 4)
    )

    print(
        "Median IoU:",
        round(np.median(all_ious), 4)
    )

    print(
        "Best IoU:",
        round(np.max(all_ious), 4)
    )

In [ ]:
from sklearn.metrics import roc_auc_score

roc_auc = roc_auc_score(
    all_labels,
    all_probabilities
)

print(f"ROC-AUC: {roc_auc:.4f}")

In [ ]:
# ============================================
# FINAL RESULTS
# ============================================

final_results = {
    "Test Accuracy": accuracy,
    "Precision": precision,
    "Recall": recall,
    "F1 Score": f1,
    "ROC-AUC": roc_auc,
    "Mean IoU": np.mean(all_ious),
    "Median IoU": np.median(all_ious),
    "Best IoU": np.max(all_ious)
}

for metric, value in final_results.items():
    print(f"{metric}: {value:.4f}")

In [ ]:
# ============================================
# GRAD-CAM THRESHOLD EXPERIMENT
# ============================================

thresholds = [0.3, 0.4, 0.5, 0.6, 0.7]

threshold_results = {}

for threshold_value in thresholds:

    threshold_ious = []

    for _, row in sample_rows.iterrows():

        patient_id_temp = row["patientId"]

        dicom_path_temp = os.path.join(
            IMAGE_DIR,
            patient_id_temp + ".dcm"
        )

        dcm_temp = pydicom.dcmread(
            dicom_path_temp
        )

        image_temp = dcm_temp.pixel_array.astype(
            np.float32
        )

        image_temp -= image_temp.min()

        if image_temp.max() > 0:
            image_temp /= image_temp.max()

        image_uint8_temp = (
            image_temp * 255
        ).astype(np.uint8)

        pil_temp = Image.fromarray(
            image_uint8_temp
        ).convert("RGB")

        input_temp = test_transform(
            pil_temp
        ).unsqueeze(0).to(device)

        model.zero_grad()

        output_temp = model(input_temp)

        prediction_temp = torch.argmax(
            output_temp,
            dim=1
        ).item()

        if prediction_temp != 1:
            continue

        cam_temp = gradcam.generate(
            input_temp,
            prediction_temp
        )

        cam_temp_resized = cv2.resize(
            cam_temp,
            (
                image_temp.shape[1],
                image_temp.shape[0]
            )
        )

        binary_temp = (
            cam_temp_resized >= threshold_value
        ).astype(np.uint8) * 255

        contours_temp, _ = cv2.findContours(
            binary_temp,
            cv2.RETR_EXTERNAL,
            cv2.CHAIN_APPROX_SIMPLE
        )

        if not contours_temp:
            continue

        largest_temp = max(
            contours_temp,
            key=cv2.contourArea
        )

        predicted_box_temp = cv2.boundingRect(
            largest_temp
        )

        gt_temp = labels[
            (labels["patientId"] == patient_id_temp) &
            (labels["Target"] == 1)
        ]

        patient_ious = []

        for _, gt_row in gt_temp.iterrows():

            gt_box_temp = (
                int(gt_row["x"]),
                int(gt_row["y"]),
                int(gt_row["width"]),
                int(gt_row["height"])
            )

            patient_ious.append(
                calculate_iou(
                    predicted_box_temp,
                    gt_box_temp
                )
            )

        if patient_ious:
            threshold_ious.append(
                max(patient_ious)
            )

    if threshold_ious:
        threshold_results[threshold_value] = np.mean(
            threshold_ious
        )

print("Mean IoU at different Grad-CAM thresholds:")

for threshold_value, mean_iou in threshold_results.items():
    print(
        f"Threshold {threshold_value}: "
        f"{mean_iou:.4f}"
    )

In [ ]:
# ============================================
# FINAL LOCALIZATION EVALUATION
# GRAD-CAM THRESHOLD = 0.7
# ============================================

BEST_THRESHOLD = 0.7

final_ious = []
final_results = []

for _, row in sample_rows.iterrows():

    patient_id_temp = row["patientId"]

    dicom_path_temp = os.path.join(
        IMAGE_DIR,
        patient_id_temp + ".dcm"
    )

    dcm_temp = pydicom.dcmread(dicom_path_temp)

    image_temp = dcm_temp.pixel_array.astype(np.float32)

    # Normalize image
    image_temp -= image_temp.min()

    if image_temp.max() > 0:
        image_temp /= image_temp.max()

    image_uint8_temp = (
        image_temp * 255
    ).astype(np.uint8)

    pil_temp = Image.fromarray(
        image_uint8_temp
    ).convert("RGB")

    input_temp = test_transform(
        pil_temp
    ).unsqueeze(0).to(device)

    # Prediction
    model.zero_grad()

    output_temp = model(input_temp)

    prediction_temp = torch.argmax(
        output_temp,
        dim=1
    ).item()

    # Only localize if predicted as Lung Opacity
    if prediction_temp != 1:
        continue

    # Generate Grad-CAM
    cam_temp = gradcam.generate(
        input_temp,
        prediction_temp
    )

    # Resize CAM to original image size
    cam_temp_resized = cv2.resize(
        cam_temp,
        (
            image_temp.shape[1],
            image_temp.shape[0]
        )
    )

    # Apply best threshold
    binary_temp = (
        cam_temp_resized >= BEST_THRESHOLD
    ).astype(np.uint8) * 255

    # Find contours
    contours_temp, _ = cv2.findContours(
        binary_temp,
        cv2.RETR_EXTERNAL,
        cv2.CHAIN_APPROX_SIMPLE
    )

    if not contours_temp:
        continue

    # Largest activated region
    largest_temp = max(
        contours_temp,
        key=cv2.contourArea
    )

    predicted_box_temp = cv2.boundingRect(
        largest_temp
    )

    # Ground-truth boxes
    gt_temp = labels[
        (labels["patientId"] == patient_id_temp) &
        (labels["Target"] == 1)
    ]

    patient_ious = []

    for _, gt_row in gt_temp.iterrows():

        gt_box_temp = (
            int(gt_row["x"]),
            int(gt_row["y"]),
            int(gt_row["width"]),
            int(gt_row["height"])
        )

        iou_temp = calculate_iou(
            predicted_box_temp,
            gt_box_temp
        )

        patient_ious.append(iou_temp)

    if patient_ious:

        best_patient_iou = max(patient_ious)

        final_ious.append(
            best_patient_iou
        )

        final_results.append({
            "patientId": patient_id_temp,
            "IoU": best_patient_iou,
            "predicted_box": predicted_box_temp
        })


# ============================================
# FINAL RESULTS
# ============================================

print("============================================")
print("FINAL GRAD-CAM LOCALIZATION RESULTS")
print("============================================")

print(f"Threshold used: {BEST_THRESHOLD}")
print(f"Cases evaluated: {len(sample_rows)}")
print(f"Successfully localized: {len(final_ious)}")

if final_ious:

    print(
        f"Mean IoU: {np.mean(final_ious):.4f}"
    )

    print(
        f"Median IoU: {np.median(final_ious):.4f}"
    )

    print(
        f"Best IoU: {np.max(final_ious):.4f}"
    )

    print(
        f"Worst IoU: {np.min(final_ious):.4f}"
    )

print("============================================")

In [ ]:
torch.save(
    model.state_dict(),
    "/kaggle/working/densenet121_frozen.pth"
)

print("Frozen model saved successfully.")

In [ ]:
# Import torchvision models and required modules

import torch
import torch.nn as nn
from torchvision import models
from torchvision.models import DenseNet121_Weights

print("Imports successful!")

In [ ]:
# ============================================
# EXPERIMENT 2 — FINE-TUNED DENSENET121
# ============================================

weights = DenseNet121_Weights.DEFAULT

model_ft = models.densenet121(weights=weights)

# Replace classifier
num_features = model_ft.classifier.in_features

model_ft.classifier = nn.Linear(
    num_features,
    2
)

# Freeze everything first
for param in model_ft.features.parameters():
    param.requires_grad = False

# Unfreeze final DenseNet block
for param in model_ft.features.denseblock4.parameters():
    param.requires_grad = True

# Unfreeze final normalization layer
for param in model_ft.features.norm5.parameters():
    param.requires_grad = True

model_ft = model_ft.to(device)

# Loss
criterion_ft = nn.CrossEntropyLoss()

# Small learning rate
optimizer_ft = torch.optim.Adam(
    filter(
        lambda p: p.requires_grad,
        model_ft.parameters()
    ),
    lr=1e-4
)

print("Fine-tuned model ready.")

trainable_params = sum(
    p.numel()
    for p in model_ft.parameters()
    if p.requires_grad
)

total_params = sum(
    p.numel()
    for p in model_ft.parameters()
)

print(f"Trainable parameters: {trainable_params:,}")
print(f"Total parameters: {total_params:,}")

In [ ]:
# ============================================
# EXPERIMENT 2 — TRAIN FINE-TUNED DENSENET121
# ============================================

num_epochs_ft = 2

for epoch in range(num_epochs_ft):

    # -------------------------
    # Training
    # -------------------------
    model_ft.train()

    running_loss = 0.0
    correct = 0
    total = 0

    for images, labels_batch in train_loader:

        images = images.to(device)
        labels_batch = labels_batch.to(device)

        optimizer_ft.zero_grad()

        outputs = model_ft(images)

        loss = criterion_ft(
            outputs,
            labels_batch
        )

        loss.backward()
        optimizer_ft.step()

        running_loss += (
            loss.item() * images.size(0)
        )

        _, predicted = torch.max(
            outputs,
            1
        )

        total += labels_batch.size(0)

        correct += (
            predicted == labels_batch
        ).sum().item()

    train_loss = running_loss / total
    train_accuracy = correct / total

    # -------------------------
    # Validation
    # -------------------------
    model_ft.eval()

    val_correct = 0
    val_total = 0

    with torch.no_grad():

        for images, labels_batch in val_loader:

            images = images.to(device)
            labels_batch = labels_batch.to(device)

            outputs = model_ft(images)

            _, predicted = torch.max(
                outputs,
                1
            )

            val_total += labels_batch.size(0)

            val_correct += (
                predicted == labels_batch
            ).sum().item()

    val_accuracy = val_correct / val_total

    print(
        f"Epoch {epoch+1}/{num_epochs_ft} | "
        f"Loss: {train_loss:.4f} | "
        f"Train Accuracy: {train_accuracy*100:.2f}% | "
        f"Validation Accuracy: {val_accuracy*100:.2f}%"
    )

In [ ]:
# ============================================
# TEST EVALUATION — FINE-TUNED MODEL
# ============================================

from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    roc_auc_score,
    confusion_matrix
)

model_ft.eval()

all_labels_ft = []
all_preds_ft = []
all_probs_ft = []

with torch.no_grad():

    for images, labels_batch in test_loader:

        images = images.to(device)
        labels_batch = labels_batch.to(device)

        outputs = model_ft(images)

        probabilities = torch.softmax(
            outputs,
            dim=1
        )

        _, predicted = torch.max(
            outputs,
            1
        )

        all_labels_ft.extend(
            labels_batch.cpu().numpy()
        )

        all_preds_ft.extend(
            predicted.cpu().numpy()
        )

        all_probs_ft.extend(
            probabilities[:, 1].cpu().numpy()
        )


# Calculate metrics
accuracy_ft = accuracy_score(
    all_labels_ft,
    all_preds_ft
)

precision_ft = precision_score(
    all_labels_ft,
    all_preds_ft
)

recall_ft = recall_score(
    all_labels_ft,
    all_preds_ft
)

f1_ft = f1_score(
    all_labels_ft,
    all_preds_ft
)

roc_auc_ft = roc_auc_score(
    all_labels_ft,
    all_probs_ft
)

cm_ft = confusion_matrix(
    all_labels_ft,
    all_preds_ft
)


print("============================================")
print("FINE-TUNED MODEL — TEST RESULTS")
print("============================================")

print(
    f"Accuracy : {accuracy_ft*100:.2f}%"
)

print(
    f"Precision: {precision_ft*100:.2f}%"
)

print(
    f"Recall   : {recall_ft*100:.2f}%"
)

print(
    f"F1 Score : {f1_ft*100:.2f}%"
)

print(
    f"ROC-AUC  : {roc_auc_ft:.4f}"
)

print("\nConfusion Matrix:")
print(cm_ft)

print("============================================")

In [ ]:
# ============================================
# SAVE FINAL MODEL
# ============================================

torch.save(
    model.state_dict(),
    "/kaggle/working/final_densenet121_pneumonia.pth"
)

print("Final model saved successfully.")

In [ ]:
# ============================================
# SAVE FINAL LOCALIZATION RESULTS
# ============================================

import pandas as pd

final_localization_df = pd.DataFrame(final_results)

final_localization_df.to_csv(
    "/kaggle/working/final_localization_results.csv",
    index=False
)

print("Localization results saved successfully.")
print(final_localization_df)

In [ ]:
# ============================================
# SORT LOCALIZATION RESULTS
# ============================================

sorted_results = sorted(
    final_results,
    key=lambda x: x["IoU"],
    reverse=True
)

print("Top 5 localization results:")

for result in sorted_results[:5]:
    print(
        result["patientId"],
        "IoU:",
        f"{result['IoU']:.4f}"
    )

print("\nBottom 5 localization results:")

for result in sorted_results[-5:]:
    print(
        result["patientId"],
        "IoU:",
        f"{result['IoU']:.4f}"
    )

# ============================================
# STEP 3 — FINAL LOCALIZATION VISUALIZATIONS
# Best / Medium / Worst Examples
# ============================================

import matplotlib.pyplot as plt
import matplotlib.patches as patches
import cv2
import os
import pydicom
import numpy as np
from PIL import Image

# --------------------------------------------
# Select examples
# --------------------------------------------

# Best
best_case = max(
    final_results,
    key=lambda x: x["IoU"]
)

# Worst
worst_case = min(
    final_results,
    key=lambda x: x["IoU"]
)

# Medium = IoU closest to median
median_iou = np.median(final_ious)

medium_case = min(
    final_results,
    key=lambda x: abs(x["IoU"] - median_iou)
)

selected_cases = [
    ("Best", best_case),
    ("Medium", medium_case),
    ("Worst", worst_case)
]

print("Selected cases:")
for name, case in selected_cases:
    print(
        f"{name}: {case['patientId']} | "
        f"IoU = {case['IoU']:.4f}"
    )


# ============================================
# Generate visualization for each case
# ============================================

for case_name, case in selected_cases:

    patient_id = case["patientId"]
    predicted_box = case["predicted_box"]

    # Load DICOM
    dicom_path = os.path.join(
        IMAGE_DIR,
        patient_id + ".dcm"
    )

    dcm = pydicom.dcmread(dicom_path)

    image = dcm.pixel_array.astype(np.float32)

    # Normalize image
    image -= image.min()

    if image.max() > 0:
        image /= image.max()

    image_uint8 = (
        image * 255
    ).astype(np.uint8)

    pil_image = Image.fromarray(
        image_uint8
    ).convert("RGB")

    # ----------------------------------------
    # Prepare model input
    # ----------------------------------------

    input_tensor = test_transform(
        pil_image
    ).unsqueeze(0).to(device)

    # ----------------------------------------
    # Generate Grad-CAM using FINAL model
    # ----------------------------------------

    model.zero_grad()

    output = model(input_tensor)

    prediction = torch.argmax(
        output,
        dim=1
    ).item()

    cam = gradcam.generate(
        input_tensor,
        prediction
    )

    # Resize CAM
    cam_resized = cv2.resize(
        cam,
        (
            image.shape[1],
            image.shape[0]
        )
    )

    # ----------------------------------------
    # Get ground-truth bounding boxes
    # ----------------------------------------

    gt_rows = labels[
        (labels["patientId"] == patient_id) &
        (labels["Target"] == 1)
    ]

    # ----------------------------------------
    # Create figure
    # ----------------------------------------

    fig, ax = plt.subplots(
        figsize=(8, 8)
    )

    ax.imshow(
        image,
        cmap="gray"
    )

    # ----------------------------------------
    # Ground truth boxes — RED
    # ----------------------------------------

    for _, gt_row in gt_rows.iterrows():

        gt_x = int(gt_row["x"])
        gt_y = int(gt_row["y"])
        gt_w = int(gt_row["width"])
        gt_h = int(gt_row["height"])

        gt_rect = patches.Rectangle(
            (gt_x, gt_y),
            gt_w,
            gt_h,
            linewidth=2,
            edgecolor="red",
            facecolor="none",
            label="Ground Truth"
        )

        ax.add_patch(gt_rect)

    # ----------------------------------------
    # Predicted Grad-CAM box — GREEN
    # ----------------------------------------

    pred_x, pred_y, pred_w, pred_h = predicted_box

    pred_rect = patches.Rectangle(
        (pred_x, pred_y),
        pred_w,
        pred_h,
        linewidth=2,
        edgecolor="lime",
        facecolor="none",
        label="Grad-CAM Prediction"
    )

    ax.add_patch(pred_rect)

    # ----------------------------------------
    # Title
    # ----------------------------------------

    ax.set_title(
        f"{case_name} Localization\n"
        f"IoU = {case['IoU']:.4f} | "
        f"Threshold = 0.7",
        fontsize=14
    )

    ax.legend(
        loc="upper right"
    )

    ax.axis("off")

    plt.tight_layout()

    # ----------------------------------------
    # Save figure
    # ----------------------------------------

    save_path = (
        f"/kaggle/working/"
        f"localization_{case_name.lower()}.png"
    )

    plt.savefig(
        save_path,
        dpi=300,
        bbox_inches="tight"
    )

    plt.show()

    print(
        f"Saved: {save_path}"
    )

# Pneumonia Diagnosis Detection and Localization

### Using DenseNet121 Transfer Learning and Grad-CAM

**Dataset:** RSNA Pneumonia Detection Challenge

**Task:** Binary classification of Normal vs Lung Opacity and weakly supervised localization using Grad-CAM.

**Model:** Pretrained DenseNet121 with frozen convolutional feature extractor

**Localization:** Grad-CAM with threshold optimization

## 1. Introduction

Pneumonia and other lung abnormalities can appear as regions of increased opacity in chest X-ray images. Automated analysis of chest radiographs can assist in identifying abnormal cases and highlighting potentially relevant regions.

This project investigates a weakly supervised approach for chest X-ray classification and localization. The model first performs binary classification between Normal and Lung Opacity cases. Grad-CAM is then used to identify image regions that contribute strongly to the model's prediction.

The project is based on a Stanford CS229 weakly supervised pneumonia localization project, with modifications including the use of a pretrained DenseNet121 architecture, transfer learning, Grad-CAM-based localization, and a Grad-CAM threshold sensitivity experiment.

## 2. Problem Statement

Develop a deep learning system that can:

1. Classify chest X-ray images as Normal or Lung Opacity.
2. Identify image regions that contribute to the predicted Lung Opacity class.
3. Compare Grad-CAM-derived localization with RSNA ground-truth bounding boxes using Intersection over Union (IoU).

## 3. Objectives

- Develop a binary chest X-ray classification model.
- Use transfer learning with DenseNet121.
- Evaluate classification performance using Accuracy, Precision, Recall, F1-score, and ROC-AUC.
- Generate weakly supervised localization maps using Grad-CAM.
- Convert Grad-CAM activation maps into bounding boxes.
- Evaluate localization using Intersection over Union (IoU).
- Investigate the effect of Grad-CAM threshold on localization performance.
- Compare frozen transfer learning with partial fine-tuning.

## 4. Dataset

The project uses the RSNA Pneumonia Detection Challenge dataset.

The dataset contains chest X-ray images along with annotations for lung opacity regions.

For this project, the classes were filtered as follows:

- Normal → class 0
- Lung Opacity → class 1
- No Lung Opacity / Not Normal → excluded from the binary experiment

Images containing multiple bounding-box annotations were deduplicated by patient ID for the classification dataset, while the original annotation table was retained for localization evaluation.

## 5. Proposed Methodology

The proposed pipeline consists of the following stages:

1. Load chest X-ray DICOM images.
2. Normalize pixel intensities.
3. Convert grayscale images to RGB.
4. Resize images to 224 × 224 pixels.
5. Apply ImageNet normalization.
6. Use pretrained DenseNet121 for feature extraction.
7. Train a binary classifier for Normal vs Lung Opacity.
8. Generate Grad-CAM activation maps for Lung Opacity predictions.
9. Threshold the activation map to obtain a localized region.
10. Extract the largest connected component as the predicted bounding box.
11. Compare the predicted bounding box with RSNA annotations using IoU.
12. Perform threshold sensitivity analysis to determine the best Grad-CAM threshold.

## 10. Final Results

### Classification

The frozen DenseNet121 model achieved:

- Accuracy: 92.67%
- Precision: 94.44%
- Recall: 90.67%
- F1-score: 92.52%
- ROC-AUC: 0.9788

### Localization

Using a Grad-CAM threshold of 0.7:

- Evaluation cases: 20
- Successfully localized: 19
- Mean IoU: 0.2477
- Median IoU: 0.2287
- Best IoU: 0.5372
- Worst IoU: 0.0356

## 11. Ablation Studies

### Frozen vs Fine-Tuned DenseNet121

| Model | Accuracy | Precision | Recall | F1-score | ROC-AUC |
|---|---:|---:|---:|---:|---:|
| Frozen DenseNet121 | 92.67% | 94.44% | 90.67% | 92.52% | 0.9788 |
| Fine-tuned DenseNet121 | 89.00% | 89.80% | 88.00% | 88.89% | 0.9597 |

The frozen DenseNet121 configuration performed better than the partial fine-tuning configuration under the experimental setup used. Therefore, the frozen model was selected as the final classification model.

### Grad-CAM Threshold Sensitivity

| Grad-CAM Threshold | Mean IoU |
|---:|---:|
| 0.3 | 0.1188 |
| 0.4 | 0.1393 |
| 0.5 | 0.1641 |
| 0.6 | 0.2066 |
| 0.7 | 0.2477 |

Mean IoU increased consistently as the Grad-CAM threshold increased from 0.3 to 0.7. Among the tested thresholds, 0.7 produced the highest mean IoU and was therefore selected for the final localization evaluation.

## 12. Limitations

- The classification experiment used a relatively small subset of the full RSNA dataset.
- Localization was evaluated on a small sample of 20 test cases.
- Grad-CAM provides coarse class-discriminative localization rather than precise object detection.
- The predicted bounding box can be substantially larger or displaced relative to the ground-truth annotation.
- Lung Opacity annotations should not be interpreted as equivalent to clinically confirmed pneumonia diagnoses.
- The experiments were performed under limited training time and computational resources.
- The localization results should therefore be considered an initial weakly supervised evaluation rather than a clinical validation.

## 13. Conclusion

This project developed a weakly supervised chest X-ray analysis pipeline using pretrained DenseNet121 and Grad-CAM.

The final frozen DenseNet121 model achieved 92.67% classification accuracy and an ROC-AUC of 0.9788 on the held-out test subset. Grad-CAM provided a mechanism for obtaining approximate abnormal-region localization without training a dedicated object detection model.

A threshold sensitivity experiment showed that increasing the Grad-CAM threshold improved mean IoU on the evaluated subset, with a threshold of 0.7 achieving a mean IoU of 0.2477.

The comparison between frozen and partially fine-tuned DenseNet121 showed that the frozen transfer-learning configuration performed better under the current experimental setup.

Overall, the results demonstrate that strong image-level classification can be achieved using transfer learning, while precise localization remains a challenging problem for weakly supervised approaches.

In [ ]:
# ============================================
# FINAL MODEL — CONFUSION MATRIX
# ============================================

from sklearn.metrics import confusion_matrix, ConfusionMatrixDisplay
import matplotlib.pyplot as plt

# Make sure final model is in evaluation mode
model.eval()

test_labels_final = []
test_predictions_final = []

with torch.no_grad():

    for images, labels_batch in test_loader:

        images = images.to(device)

        outputs = model(images)

        predictions = torch.argmax(
            outputs,
            dim=1
        )

        test_labels_final.extend(
            labels_batch.numpy()
        )

        test_predictions_final.extend(
            predictions.cpu().numpy()
        )

# Confusion matrix
cm_final = confusion_matrix(
    test_labels_final,
    test_predictions_final
)

print("Final Confusion Matrix:")
print(cm_final)

# Display
disp = ConfusionMatrixDisplay(
    confusion_matrix=cm_final,
    display_labels=[
        "Normal",
        "Lung Opacity"
    ]
)

fig, ax = plt.subplots(
    figsize=(6, 6)
)

disp.plot(
    ax=ax,
    cmap="Blues",
    values_format="d"
)

ax.set_title(
    "DenseNet121 Classification Confusion Matrix"
)

plt.tight_layout()

# Save
plt.savefig(
    "/kaggle/working/confusion_matrix.png",
    dpi=300,
    bbox_inches="tight"
)

plt.show()

In [ ]:
# ============================================
# FINAL MODEL — ROC CURVE
# ============================================

from sklearn.metrics import (
    roc_curve,
    roc_auc_score
)
import matplotlib.pyplot as plt

# Final model
model.eval()

roc_labels = []
roc_probabilities = []

with torch.no_grad():

    for images, labels_batch in test_loader:

        images = images.to(device)

        outputs = model(images)

        probabilities = torch.softmax(
            outputs,
            dim=1
        )

        # Probability of Lung Opacity
        lung_opacity_prob = probabilities[:, 1]

        roc_labels.extend(
            labels_batch.numpy()
        )

        roc_probabilities.extend(
            lung_opacity_prob.cpu().numpy()
        )


# Calculate ROC curve
fpr, tpr, thresholds_roc = roc_curve(
    roc_labels,
    roc_probabilities
)

roc_auc_final = roc_auc_score(
    roc_labels,
    roc_probabilities
)

print(
    f"Final ROC-AUC: {roc_auc_final:.4f}"
)


# ============================================
# Plot ROC Curve
# ============================================

plt.figure(
    figsize=(7, 6)
)

plt.plot(
    fpr,
    tpr,
    linewidth=2,
    label=f"DenseNet121 (AUC = {roc_auc_final:.4f})"
)

plt.plot(
    [0, 1],
    [0, 1],
    linestyle="--",
    linewidth=1.5,
    label="Random Classifier"
)

plt.xlabel(
    "False Positive Rate"
)

plt.ylabel(
    "True Positive Rate"
)

plt.title(
    "ROC Curve — Normal vs Lung Opacity"
)

plt.legend(
    loc="lower right"
)

plt.grid(
    alpha=0.3
)

plt.tight_layout()

plt.savefig(
    "/kaggle/working/roc_curve.png",
    dpi=300,
    bbox_inches="tight"
)

plt.show()

In [ ]:
# ============================================
# TRAINING AND VALIDATION ACCURACY COMPARISON
# ============================================

import matplotlib.pyplot as plt

# Frozen DenseNet121
epochs_frozen = [1, 2]
train_acc_frozen = [91.71, 93.29]
val_acc_frozen = [93.33, 93.33]

# Fine-tuned DenseNet121
epochs_ft = [1, 2]
train_acc_ft = [83.14, 90.64]
val_acc_ft = [87.67, 91.67]


# ============================================
# Plot
# ============================================

plt.figure(figsize=(8, 6))

plt.plot(
    epochs_frozen,
    train_acc_frozen,
    marker="o",
    linewidth=2,
    label="Frozen - Train"
)

plt.plot(
    epochs_frozen,
    val_acc_frozen,
    marker="o",
    linewidth=2,
    label="Frozen - Validation"
)

plt.plot(
    epochs_ft,
    train_acc_ft,
    marker="s",
    linewidth=2,
    label="Fine-tuned - Train"
)

plt.plot(
    epochs_ft,
    val_acc_ft,
    marker="s",
    linewidth=2,
    label="Fine-tuned - Validation"
)

plt.xlabel("Epoch")
plt.ylabel("Accuracy (%)")

plt.title(
    "Training and Validation Accuracy Comparison"
)

plt.xticks([1, 2])

plt.ylim(
    75,
    100
)

plt.grid(
    alpha=0.3
)

plt.legend()

plt.tight_layout()

plt.savefig(
    "/kaggle/working/training_validation_accuracy.png",
    dpi=300,
    bbox_inches="tight"
)

plt.show()

In [ ]:
# ============================================
# TRAINING LOSS COMPARISON
# ============================================

# Recorded training losses

loss_frozen = [
    0.1940,
    0.1716
]

loss_finetuned = [
    0.3843,
    0.2421
]

epochs = [1, 2]


plt.figure(figsize=(8, 6))

plt.plot(
    epochs,
    loss_frozen,
    marker="o",
    linewidth=2,
    label="Frozen DenseNet121"
)

plt.plot(
    epochs,
    loss_finetuned,
    marker="s",
    linewidth=2,
    label="Fine-tuned DenseNet121"
)

plt.xlabel("Epoch")
plt.ylabel("Training Loss")

plt.title(
    "Training Loss Comparison"
)

plt.xticks([1, 2])

plt.grid(
    alpha=0.3
)

plt.legend()

plt.tight_layout()

plt.savefig(
    "/kaggle/working/training_loss_comparison.png",
    dpi=300,
    bbox_inches="tight"
)

plt.show()

## 14. Final Results Summary

### Classification Performance

| Metric | Frozen DenseNet121 |
|---|---:|
| Accuracy | 92.67% |
| Precision | 94.44% |
| Recall | 90.67% |
| F1-score | 92.52% |
| ROC-AUC | 0.9788 |

### Localization Performance

| Metric | Result |
|---|---:|
| Grad-CAM threshold | 0.7 |
| Cases evaluated | 20 |
| Successfully localized | 19 |
| Mean IoU | 0.2477 |
| Median IoU | 0.2287 |
| Best IoU | 0.5372 |
| Worst IoU | 0.0356 |

### Model Ablation

| Model | Accuracy | Precision | Recall | F1-score | ROC-AUC |
|---|---:|---:|---:|---:|---:|
| Frozen DenseNet121 | 92.67% | 94.44% | 90.67% | 92.52% | 0.9788 |
| Fine-tuned DenseNet121 | 89.00% | 89.80% | 88.00% | 88.89% | 0.9597 |

## 15. Comparison with the Reference Project

The reference Stanford CS229 project used a custom CNN and Class Activation Mapping (CAM) for weakly supervised pneumonia localization.

The present project modifies the approach by:

- Using pretrained DenseNet121 instead of a custom CNN.
- Using transfer learning for image classification.
- Using Grad-CAM instead of the original CAM approach.
- Performing Grad-CAM threshold sensitivity analysis.
- Evaluating localization using IoU.
- Comparing frozen transfer learning with partial fine-tuning.

The reference project reported approximately 92.47% classification accuracy and an IoU-based localization evaluation. Since the datasets, preprocessing, training configuration, and evaluation subsets are not identical, the numerical results should not be interpreted as a direct apples-to-apples comparison.

## 16. Discussion

The results demonstrate that DenseNet121 can provide strong image-level discrimination between Normal and Lung Opacity chest X-rays. The final model achieved 92.67% accuracy and an ROC-AUC of 0.9788 on the held-out test subset.

However, image classification and precise localization are different tasks. Although the classifier performed strongly, Grad-CAM localization was considerably less precise, with a mean IoU of 0.2477 on the evaluated subset.

The threshold experiment showed that localization improved consistently as the Grad-CAM threshold increased from 0.3 to 0.7. This indicates that suppressing weaker activations helped produce more focused localization regions for this evaluation.

The fine-tuning experiment produced lower test performance than the frozen DenseNet121 configuration. This suggests that, under the current dataset size and training configuration, the pretrained features were more effective than partially adapting the final DenseNet block.

The qualitative examples further demonstrate the behavior of weakly supervised localization: some cases show substantial overlap with the annotated regions, while other cases produce broad or displaced bounding boxes.

## 17. Future Work

Several improvements could be explored in future work:

- Train on a substantially larger portion of the RSNA dataset.
- Perform patient-level train/validation/test splitting on the complete dataset.
- Evaluate localization on a larger test population.
- Investigate multi-scale Grad-CAM or other explainability methods.
- Compare Grad-CAM with dedicated object-detection or segmentation approaches.
- Explore more extensive DenseNet fine-tuning with optimized learning rates and regularization.
- Investigate segmentation-based localization for more precise abnormality boundaries.
- Evaluate the system on external chest X-ray datasets to assess generalization.

## 18. Conclusion

This project developed a weakly supervised chest X-ray analysis system for Normal versus Lung Opacity classification and approximate abnormal-region localization.

A pretrained DenseNet121 with a frozen feature extractor was selected as the final classification model. It achieved 92.67% accuracy, 94.44% precision, 90.67% recall, 92.52% F1-score, and 0.9788 ROC-AUC on the held-out test subset.

Grad-CAM was used to obtain weakly supervised localization without requiring a separately trained object-detection model. A threshold sensitivity experiment identified 0.7 as the best threshold among the tested values, producing a mean IoU of 0.2477 across 19 successfully localized cases from a 20-case evaluation subset.

The experiments also showed that partial fine-tuning of DenseNet121 did not improve performance under the current training configuration. Overall, the project demonstrates the effectiveness of transfer learning for chest X-ray classification while highlighting the limitations of weakly supervised methods for precise localization.

## Dataset and Preprocessing

We use the RSNA Pneumonia Detection Challenge dataset containing chest X-ray images in DICOM format.

For this binary classification experiment, the classes were defined as:

- Normal → Class 0
- Lung Opacity → Class 1

The No Lung Opacity / Not Normal category was excluded.

A subset of 2,000 images was used for the experiment, consisting of:
- 1,000 Normal images
- 1,000 Lung Opacity images

The dataset was divided into:
- 1,400 training images
- 300 validation images
- 300 test images

The DICOM images were normalized, converted from grayscale to RGB, and resized to 224 × 224 pixels. ImageNet normalization was applied before feeding the images into DenseNet121.

Training images were augmented using random horizontal flipping and small rotations.